# Demo — RAG agêntico: uma pergunta, várias buscas

**Objetivo:** reproduzir o **degrau 3** do slide 17. O sintoma: falha em pergunta que junta dois assuntos ou exige mais de uma fonte. O remédio: "Agentic retrieval planeja subconsultas". É a linha **Agêntico** do slide 15: planeja subconsultas e busca em paralelo, resolve pergunta composta e fontes múltiplas, e custa várias chamadas. Fecha com a regra do slide 13: subir de degrau só por sintoma medido.

1. **A pergunta composta numa busca só** → uma consulta híbrida com reranking devolve uma lista só, e um dos assuntos pode ficar de fora;
2. **Planejar as subconsultas** → o Azure OpenAI divide a pergunta em buscas curtas, uma por assunto;
3. **Agentic retrieval no AI Search** → o knowledge base executa as subconsultas em paralelo, com semantic ranker, e junta os resultados;
4. **Resposta final com citação** → a geração recebe trechos dos dois assuntos, e contamos quantas chamadas isso custou.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** o índice `cap05-acervo`, criado e carregado pelo notebook `tipos-de-rag/00-indice-compartilhado`. Este notebook cria, no mesmo serviço, o knowledge source `cap05-ks-acervo` e o knowledge base `cap05-kb-acervo`.

## 0. Preparação

Este notebook usa a **Azure OpenAI v1 API** (cliente Python `OpenAI`) para planejar as subconsultas e gerar a resposta, e o SDK `azure-search-documents` 12, da API GA `2026-04-01`, para criar o knowledge source e o knowledge base e chamar o retrieve.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat: planejamento na Execução 2 e resposta na Execução 4). Ele precisa aceitar **structured outputs**, porque o plano volta num formato fixo; o `gpt-5-mini` sugerido no `.env.example` aceita.
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os **mesmos** valores usados para criar o índice no notebook 00
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY` (chave de **administração**: este notebook cria o knowledge source e o knowledge base)

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

O serviço do AI Search precisa estar numa região com agentic retrieval (Brazil South tem). O agentic retrieval é cobrado por token: o plano Free, que é o padrão, dá uma cota mensal gratuita. Esgotada a cota, o retrieve devolve erro de pagamento até alguém ativar o plano Standard em **Settings > Premium features > Knowledge retrieval**, que exige o tier Basic ou superior.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "pydantic>=2.7" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

NOME_INDICE = "cap05-acervo"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

# Criados na Execução 3, sobre o mesmo índice. A limpeza, no fim, apaga os dois e mantém o índice.
NOME_KS = "cap05-ks-acervo"
NOME_KB = "cap05-kb-acervo"

# O índice é criado e carregado pelo notebook 00.
if NOME_INDICE not in list(index_client.list_index_names()):
    raise RuntimeError("Índice não encontrado. Rode antes o notebook tipos-de-rag/00-indice-compartilhado.")

print(f"Índice '{NOME_INDICE}' encontrado, com {search_client.get_document_count()} trechos.")

### Funções auxiliares

- `gerar_embeddings`: a mesma do notebook 00. Na Execução 1, a pergunta vira vetor no cliente, como nos notebooks 01 a 03.
- `responder`: a geração com citação dos notebooks anteriores, usada na Execução 4.
- `PERGUNTAS` e `cobertura`: o gabarito (`acervo/perguntas.json`) diz quais trechos respondem a cada assunto da pergunta composta, e `cobertura` conta quantos assuntos apareceram numa lista de trechos (por exemplo, `1/2`).
- `assunto_do_trecho` e `assuntos_fora`: dão nome aos dois assuntos, para as tabelas mostrarem qual deles cada trecho cobre e qual ficou de fora.

In [ ]:
import json

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text


def cobertura(ids_retornados: list[str], grupos: list[list[str]]) -> str:
    """Quantos dos assuntos esperados (grupos de trechos) apareceram na lista retornada."""
    achados = sum(any(i in ids_retornados for i in grupo) for grupo in grupos)
    return f"{achados}/{len(grupos)}"


# A pergunta composta do gabarito e o nome de cada assunto, na ordem dos grupos de trechos esperados.
COMPOSTA = PERGUNTAS["pergunta-composta"]
ASSUNTOS = ["teto da diária de hotel", "NR-12 rev. 3 antes de abrir a proteção"]


def assunto_do_trecho(id_trecho: str) -> str:
    """O assunto da pergunta composta que este trecho cobre, ou vazio se ele não cobre nenhum."""
    for nome, grupo in zip(ASSUNTOS, COMPOSTA["trechos_esperados"]):
        if id_trecho in grupo:
            return nome
    return ""


def assuntos_fora(ids_retornados: list[str]) -> list[str]:
    """Os assuntos da pergunta composta que não têm nenhum trecho na lista."""
    cobertos = {assunto_do_trecho(i) for i in ids_retornados}
    return [nome for nome in ASSUNTOS if nome not in cobertos]

---

# Execução 1 — A pergunta composta numa busca só

A pergunta do gabarito junta **dois assuntos de dois documentos**: o teto da diária de hotel, na política de viagens, e o que a NR-12 rev. 3 exige antes de abrir a proteção da máquina, no procedimento de segurança.

Rodamos o melhor pipeline de uma busca só, o do notebook 03: consulta **híbrida** (BM25 e vetor, fundidos por RRF) reordenada pelo **semantic ranker**. O lado vetorial pede `k=50`, como recomenda a documentação, porque o reranker lê até 50 candidatos. Ficam os 5 primeiros: seriam o contexto da geração (slide 16).

### O que esperar e por que pode variar

- Uma busca só devolve **uma lista só**, ordenada pela relevância para a pergunta inteira. Os trechos de prensa, manutenção e proteção de máquina se parecem com a maior parte da pergunta e tendem a ocupar as cinco vagas.
- Neste acervo pequeno, os dois assuntos podem caber no top 5, e a cobertura dá `2/2`. Nesse caso, anote a nota do reranker da linha cujo assunto é o teto da diária de hotel: na Execução 3, você vai compará-la com a nota do mesmo trecho na subconsulta focada.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def buscar_uma_vez(consulta: str, k: int = 5) -> list[dict]:
    """Uma consulta híbrida (BM25 + vetor, fusão RRF), reordenada pelo semantic ranker."""
    vetor = gerar_embeddings([consulta])[0]
    resultados = search_client.search(
        search_text=consulta,
        # O reranker lê até 50 candidatos: o lado vetorial precisa entregar 50.
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=50, fields="embedding")],
        query_type="semantic",
        semantic_configuration_name="semantica-padrao",
        semantic_error_mode="partial",  # se o reranker falhar, volta a ordem do híbrido, sem nota
        select=["id", "titulo", "secao"],
        top=k,
    )
    return [
        {"id": r["id"], "titulo": r["titulo"], "secao": r["secao"], "nota_reranker": r["@search.reranker_score"]}
        for r in resultados
    ]


top5 = buscar_uma_vez(COMPOSTA["pergunta"], k=5)
ids_top5 = [t["id"] for t in top5]

display(Markdown(f"**Pergunta:** {COMPOSTA['pergunta']}"))
display(pd.DataFrame([
    {
        "posição": i,
        "nota do reranker": t["nota_reranker"],
        "trecho": f"{t['titulo']} — {t['secao']}",
        "assunto da pergunta": assunto_do_trecho(t["id"]),
    }
    for i, t in enumerate(top5, start=1)
]).round(2))

fora = assuntos_fora(ids_top5)
display(Markdown(
    f"**Cobertura dos assuntos no top 5:** {cobertura(ids_top5, COMPOSTA['trechos_esperados'])}  \n"
    f"**Ficou de fora:** {', '.join(fora) if fora else 'nenhum assunto'}"
))

### O que observar

- A coluna **assunto da pergunta** mostra quem ocupou o top 5. Quando a coluna fica vazia, o trecho, segundo o gabarito, não responde a nenhuma das duas partes da pergunta, mas tomou uma vaga.
- A **cobertura** conta assuntos, não trechos: `1/2` quer dizer que o contexto da geração não traria a informação de um dos assuntos, por melhor que seja o modelo.
- Aumentar o top até caber tudo funciona num acervo de 50 trechos, mas é o anti-padrão "mandar vinte trechos ao modelo" (slide 28). O remédio é buscar cada assunto separadamente.

### Fala sugerida

> “A pergunta tem dois assuntos, mas a busca devolve uma lista só. É o sintoma do degrau 3 do slide 17: falha em pergunta que junta dois assuntos ou exige mais de uma fonte.”

---

# Execução 2 — Planejar as subconsultas

Na API GA do agentic retrieval (`2026-04-01`), o knowledge base **não planeja**: ele recebe as subconsultas prontas, que a API chama de *intents*. Quem planeja, então, é o app, com uma chamada ao Azure OpenAI.

A chamada usa **saída estruturada**: `responses.parse` com um modelo Pydantic (`Plano`, uma lista de `Subconsulta`) faz a resposta voltar como dados, prontos para virar intents, e não como texto livre. As instruções pedem de 2 a 4 consultas curtas e independentes, em português, com os códigos e nomes exatos, como `NR-12 rev. 3` e `P-400`: o BM25 só acha o código se ele chegar inteiro.

In [ ]:
from pydantic import BaseModel


class Subconsulta(BaseModel):
    consulta: str
    objetivo: str


class Plano(BaseModel):
    subconsultas: list[Subconsulta]


INSTRUCOES_PLANO = """
Você planeja buscas num acervo de políticas, normas e procedimentos internos.
Divida a pergunta do usuário em 2 a 4 consultas de busca curtas e independentes, em português.
Cada consulta trata de um só assunto e precisa fazer sentido sozinha.
Mantenha exatamente como estão os códigos, nomes e versões, como NR-12 rev. 3 e P-400.
Em objetivo, diga em poucas palavras que parte da pergunta a consulta responde.
""".strip()

resposta_plano = client.responses.parse(
    model=AZURE_OPENAI_DEPLOYMENT,
    instructions=INSTRUCOES_PLANO,
    input=COMPOSTA["pergunta"],
    text_format=Plano,
    store=False,
)
plano = resposta_plano.output_parsed

display(Markdown(f"**Pergunta:** {COMPOSTA['pergunta']}"))
display(pd.DataFrame([
    {"nº": i, "consulta": s.consulta, "objetivo": s.objetivo}
    for i, s in enumerate(plano.subconsultas, start=1)
]))

### O que observar

- Cada subconsulta deve tratar de **um** assunto: uma do teto da diária de hotel, outra da NR-12 rev. 3 antes de abrir a proteção. O modelo pode criar uma terceira, por exemplo sobre a manutenção da P-400; o plano varia de uma execução para outra.
- Confira se os códigos ficaram intactos. Uma subconsulta que troca "NR-12 rev. 3" por "norma de segurança de máquinas" perde justamente o que o BM25 sabe achar.

### Leitura arquitetural

- **Na API GA, o app planeja e o serviço executa.** O planejamento é uma chamada a mais ao Azure OpenAI, com custo e latência próprios, e fica no seu código: dá para versionar o prompt, registrar o plano e testá-lo.
- **No preview (`2026-08-01-preview`), o knowledge base também pode planejar as subconsultas sozinho**, a partir da conversa (`messages`), com um modelo do Azure OpenAI configurado nele. É esse o modo que o slide 17 descreve: "Agentic retrieval planeja subconsultas".

---

# Execução 3 — Agentic retrieval no AI Search

O agentic retrieval usa dois objetos no serviço, os dois apontando para o índice que já existe:

- **Knowledge source** `cap05-ks-acervo`: diz qual índice consultar, com qual configuração semântica (`semantica-padrao`, obrigatória na API `2026-04-01`) e quais campos devolver em cada referência (`source_data_fields`).
- **Knowledge base** `cap05-kb-acervo`: o objeto que recebe o retrieve e orquestra as fontes. Na API GA, ele **não leva modelo** (`models`): é só a lista de knowledge sources.

No retrieve, cada subconsulta do plano vira um *intent*. O serviço roda cada intent como consulta **híbrida** (texto e vetor: o campo `embedding` tem o vetorizador `vetorizador-aoai` do notebook 00, que vetoriza a subconsulta no seu Azure OpenAI), reordena com o **semantic ranker**, executa os intents **em paralelo** e junta os melhores trechos. A resposta traz a **atividade** (o que o serviço executou) e as **referências** (os trechos, com a nota do reranker).

### O que esperar e por que pode variar

- O plano muda a cada execução, e com ele as subconsultas e os trechos devolvidos.
- O notebook fixa o corte da nota do reranker em 2,5, numa escala de 0 a 4 (`reranker_threshold`): é o valor que a documentação descreve, a nota mínima para um trecho entrar na resposta. O retrieve também limita a saída a 5.000 tokens por padrão (`max_output_size_in_tokens`). A coluna **trechos acima do corte** da atividade conta quantos trechos passaram em cada subconsulta: se um assunto não aparecer, comece por ela.
- `create_or_update` deixa a célula segura para rodar de novo: os dois objetos são atualizados, não duplicados.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

knowledge_source = SearchIndexKnowledgeSource(
    name=NOME_KS,
    description="Políticas, normas e procedimentos fictícios da Contoso Industrial (índice cap05-acervo).",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=NOME_INDICE,
        semantic_configuration_name="semantica-padrao",
        # Os campos que voltam em cada referência: com eles montamos o contexto e citamos a fonte.
        source_data_fields=[SearchIndexFieldReference(name=c) for c in ["id", "titulo", "secao", "conteudo", "fonte"]],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source)

# Na API GA, o knowledge base é só a lista de fontes: sem modelo, sem planejamento pelo LLM.
knowledge_base = KnowledgeBase(
    name=NOME_KB,
    description="Knowledge base do Cap. 5 sobre o acervo fictício da Contoso Industrial.",
    knowledge_sources=[KnowledgeSourceReference(name=NOME_KS)],
)
kb_registrado = index_client.create_or_update_knowledge_base(knowledge_base)

print(f"Knowledge source '{NOME_KS}' aponta para o índice '{NOME_INDICE}'.")
print(f"Knowledge base '{NOME_KB}', como o serviço o registrou:")
print(json.dumps(kb_registrado.as_dict(), indent=2, ensure_ascii=False))

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)

# O nome do knowledge base vai como argumento nomeado: na versão 12 do SDK, a forma posicional dá TypeError.
kb_client = KnowledgeBaseRetrievalClient(AZURE_SEARCH_ENDPOINT, credencial_busca, knowledge_base_name=NOME_KB)

# Uma chamada só: o serviço executa as subconsultas do plano em paralelo e junta os resultados.
resultado = kb_client.retrieve(KnowledgeBaseRetrievalRequest(
    intents=[KnowledgeRetrievalSemanticIntent(search=s.consulta) for s in plano.subconsultas],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=NOME_KS,
            include_references=True,
            include_reference_source_data=True,
            reranker_threshold=2.5,  # o corte da nota do reranker: suba para menos trechos, desça para mais
        ),
    ],
    include_activity=True,
))

# A atividade: um registro por etapa executada. Os campos variam por tipo de registro; as_dict() traz todos.
linhas_atividade = []
for registro in resultado.activity or []:
    a = registro.as_dict()
    linhas_atividade.append({
        "id": a.get("id"),
        "tipo": a.get("type"),
        "subconsulta executada": (a.get("searchIndexArguments") or {}).get("search", ""),
        "trechos acima do corte": a.get("count", ""),
        "tempo (ms)": a.get("elapsedMs", ""),
        "tokens (reasoningTokens)": a.get("reasoningTokens", ""),
        "erro": (a.get("error") or {}).get("message", ""),
    })
display(Markdown("**Atividade:** o que o serviço executou"))
display(pd.DataFrame(linhas_atividade))

# As referências: os trechos que o serviço juntou. doc_key é a chave do trecho no índice.
referencias = resultado.references or []
ids_kb = [r.doc_key for r in referencias]
display(Markdown(f"**Referências:** {len(referencias)} trechos"))
display(pd.DataFrame([
    {
        "id do trecho": r.doc_key,
        "seção": (r.source_data or {}).get("secao", ""),
        "nota do reranker": r.reranker_score,
        "atividade de origem": r.activity_source,
        "assunto da pergunta": assunto_do_trecho(r.doc_key),
    }
    for r in referencias
]).round(2))

fora = assuntos_fora(ids_kb)
display(Markdown(
    f"**Cobertura dos assuntos:** {cobertura(ids_kb, COMPOSTA['trechos_esperados'])}  \n"
    f"**Ficou de fora:** {', '.join(fora) if fora else 'nenhum assunto'}"
))

### O que observar

- Na **atividade**, cada linha `searchIndex` é uma subconsulta do plano, executada pelo serviço, com o tempo dela e quantos trechos passaram do corte. A linha `agenticReasoning` traz a contagem de tokens da recuperação: o agentic retrieval é cobrado por token, não por consulta.
- Nas **referências**, a coluna **atividade de origem** aponta para o `id` da linha da atividade que achou o trecho: dá para ver qual subconsulta trouxe qual assunto.
- Compare a nota do trecho do teto da diária (`financeiro_reembolso-de-despesas-de-viagem_02`) com a da Execução 1, se ele apareceu lá: na subconsulta focada, ele responde à consulta inteira, e o reranker tende a dar nota maior.
- A **cobertura** tende a chegar a `2/2`: cada assunto teve a sua própria busca e as suas próprias vagas.
- O retrieve também devolve `response`: um texto JSON único com os trechos mais relevantes, pronto para ir ao prompt. Aqui usamos as referências, porque elas trazem a chave de cada trecho no índice (`doc_key`), que é o que a citação usa. A documentação descreve o corte de 2,5 para esse texto, mas não deixa claro se ele vale também para as referências: confira na coluna **nota do reranker** se alguma ficou abaixo.

### Leitura arquitetural

- **No preview**, além de planejar as subconsultas (Execução 2), o knowledge base também pode sintetizar a resposta. Recurso em preview não tem SLA.
- O índice é o mesmo dos notebooks 01 a 03. O que muda é o modelo de recuperação: objetos próprios (knowledge source e knowledge base) e outra API (retrieve).

### Fala sugerida

> “No Azure AI Search os três primeiros são configuração do mesmo índice. O agêntico é outro modelo de recuperação, com planejamento e fusão.”

---

# Execução 4 — Resposta final com citação, e quanto ela custou

Os trechos das referências viram o contexto da mesma função `responder` dos notebooks anteriores, que cita cada afirmação no formato `[id do trecho]`. Duas subconsultas podem achar o mesmo trecho, então cada trecho entra uma vez só.

Depois contamos as operações dos dois caminhos: a busca única da Execução 1, seguida da geração, e o caminho agêntico. É a coluna **Custo** do slide 15: "Várias chamadas".

### O que esperar e por que pode variar

- Com os dois assuntos no contexto, a resposta tende a trazer o teto da diária (R$ 320, porque Joinville não está na lista de capitais da política) e o bloqueio de todas as energias que a NR-12 rev. 3 exige antes de abrir a proteção, citando um trecho de cada documento.
- O texto muda a cada execução. O que importa é conferir se cada parte da resposta cita o trecho do assunto certo.

In [ ]:
def trecho_da_referencia(referencia) -> dict:
    """Uma referência do knowledge base no formato de trecho que responder() espera."""
    dados = referencia.source_data or {}
    return {
        "id": referencia.doc_key,
        "titulo": dados.get("titulo", ""),
        "secao": dados.get("secao", ""),
        "conteudo": dados.get("conteudo", ""),
    }


trechos_kb = []
for referencia in sorted(referencias, key=lambda r: r.reranker_score or 0, reverse=True):
    if referencia.doc_key not in [t["id"] for t in trechos_kb]:  # o mesmo trecho pode vir de duas subconsultas
        trechos_kb.append(trecho_da_referencia(referencia))

display(Markdown(f"### {COMPOSTA['pergunta']}"))
display(Markdown("**Trechos enviados ao modelo:** " + ", ".join(f"`{t['id']}`" for t in trechos_kb)))
display(Markdown(responder(COMPOSTA["pergunta"], trechos_kb)))

# O custo: N é o número de subconsultas que o serviço executou (as linhas searchIndex da atividade).
n = sum(1 for registro in resultado.activity or [] if registro.type == "searchIndex") or len(plano.subconsultas)
display(Markdown("**Operações de cada caminho**"))
display(pd.DataFrame(
    [
        ["Planejar as subconsultas (Azure OpenAI, chat)", 0, 1],
        ["Vetorizar a consulta (Azure OpenAI, embedding)", 1, n],
        ["Consulta híbrida com semantic ranker (AI Search)", 1, n],
        ["Gerar a resposta (Azure OpenAI, chat)", 1, 1],
        ["Total de operações", 3, 2 * n + 2],
        ["Trechos no contexto da geração", len(top5), len(trechos_kb)],
    ],
    columns=["etapa", "uma busca só", "agêntico"],
))
uso = resposta_plano.usage
print(f"Só o planejamento consumiu {uso.input_tokens} tokens de entrada e {uso.output_tokens} de saída.")

### O que observar

- Antes de julgar a resposta, confira os **trechos enviados**: há pelo menos um de cada assunto? Se faltar um, o erro nasceu na recuperação, não no modelo (slide 29).
- Na tabela de operações, o agêntico troca **uma** busca por **N**, soma o planejamento e, em geral, manda mais trechos para a geração. A vetorização das subconsultas é feita pelo vetorizador do índice, no seu Azure OpenAI, e não aparece no código.

### Leitura arquitetural

- O agêntico resolve pergunta composta e fontes múltiplas, mas custa várias chamadas (slide 15) e acrescenta latência. É o degrau 3 do slide 17: para quando o sintoma foi medido, não o padrão.
- A medida é a de sempre: perguntas com resposta conhecida, agora com a cobertura por assunto. Se as perguntas compostas do seu gabarito já chegam a `2/2` com híbrido e reranking, o agêntico só acrescenta custo.

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Uma busca só devolve uma lista só, e um dos dois assuntos pode ficar fora do top 5 | Medir a cobertura por assunto, não só a posição do primeiro trecho |
| 2 | O Azure OpenAI divide a pergunta em subconsultas, com saída estruturada | Na API GA, o planejamento é do app: prompt versionado e plano registrado |
| 3 | O knowledge base executa as subconsultas em paralelo, com semantic ranker, e junta os resultados | O agêntico é outro modelo de recuperação, sobre o mesmo índice |
| 4 | A resposta tende a citar os dois documentos, ao custo de várias chamadas | Subir para agêntico só quando o sintoma for medido |

## Frase de fechamento

> **Comece em híbrido com reranking. Suba para agêntico ou grafo por sintoma medido, nunca por moda.**

---

# Referências oficiais

- Azure AI Search — visão geral do agentic retrieval (GA e preview):  
  https://learn.microsoft.com/azure/search/agentic-retrieval-overview

- Azure AI Search — criar um knowledge base:  
  https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base

- Azure AI Search — knowledge source sobre um índice existente:  
  https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-search-index

- Azure AI Search — consultar um knowledge base com o retrieve:  
  https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve

---

# Limpeza (opcional)

O knowledge base e o knowledge source são definições que apontam para o índice `cap05-acervo`: apagá-los não apaga o índice. A ordem importa: primeiro o knowledge base, depois o knowledge source, porque o serviço recusa apagar um knowledge source que ainda é usado por um knowledge base.

O índice continua de pé: quem o apaga é a limpeza do notebook 00, depois desta.

In [ ]:
APAGAR_RECURSOS = False  # o índice cap05-acervo não é apagado aqui: ele é do notebook 00

if APAGAR_RECURSOS:
    index_client.delete_knowledge_base(NOME_KB)  # primeiro o knowledge base, que usa o knowledge source
    index_client.delete_knowledge_source(NOME_KS)
    print(f"Knowledge base '{NOME_KB}' e knowledge source '{NOME_KS}' apagados. O índice '{NOME_INDICE}' continua.")
else:
    print(f"Nada foi apagado. '{NOME_KB}' e '{NOME_KS}' continuam no serviço, apontando para '{NOME_INDICE}'.")